In [1]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, auc
)
from torch_geometric.loader import NeighborLoader
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch

def topk_lift(y_true, y_prob, k_frac=0.05):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    k = max(1, int(np.ceil(n * k_frac)))
    top_idx = np.argsort(-y_prob)[:k]
    base_rate = y_true.mean() if n > 0 else 0.0
    precision_at_k = y_true[top_idx].mean() if k > 0 else 0.0
    lift = (precision_at_k / base_rate) if base_rate > 0 else 0.0
    return int(k), float(precision_at_k), float(base_rate), float(lift)

test_loader = NeighborLoader(
    data_graph,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE * 2,
    input_nodes=('customer', data_graph['customer'].test_mask),
    shuffle=False, num_workers=0,
)

model_sage.eval()
all_preds, all_labels, all_probs_test = [], [], []

with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        edge_weight_dict = {et: b[et].edge_weight for et in b.edge_types if 'edge_weight' in b[et]}
        edge_attr_dict = {et: b[et].edge_attr for et in b.edge_types if 'edge_attr' in b[et]}
        out_t = model_sage(b.x_dict, b.edge_index_dict, edge_weight_dict, edge_attr_dict)
        bs_t = b['customer'].batch_size
        probs_t = torch.sigmoid(out_t.squeeze()[:bs_t]).cpu()
        y_t = b['customer'].y[:bs_t].cpu()
        lm_t = y_t != -1
        if lm_t.sum() > 0:
            all_probs_test.extend(probs_t[lm_t].tolist())
            all_preds.extend((probs_t[lm_t] > 0.5).float().tolist())
            all_labels.extend(y_t[lm_t].tolist())

all_probs_test = np.array(all_probs_test)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(f"Test: {len(all_labels):,} ({int(all_labels.sum())} fraud, {int((all_labels==0).sum())} legit)\n")
print(classification_report(all_labels, all_preds, target_names=['Legit', 'Fraud'], digits=4))

cm = confusion_matrix(all_labels, all_preds)
print(f"Confusion Matrix:\n  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")

prec_v, rec_v, _ = precision_recall_curve(all_labels, all_probs_test)
pr_auc = auc(rec_v, prec_v)

fr = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1]+cm[1,0])>0 else 0
fp2 = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1]+cm[0,1])>0 else 0
ff1 = 2*fr*fp2/(fr+fp2) if (fr+fp2)>0 else 0

print(f"\nPR-AUC={pr_auc:.4f}")
print(f"Fraud Recall={fr:.4f}  Fraud Precision={fp2:.4f}  F1={ff1:.4f}")

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<10} {'TP'}")
print("-"*55)
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pt = (all_probs_test > thr).astype(int)
    tp = ((pt==1)&(all_labels==1)).sum()
    fp_ = ((pt==1)&(all_labels==0)).sum()
    fn_ = ((pt==0)&(all_labels==1)).sum()
    pr_ = tp/(tp+fp_) if (tp+fp_)>0 else 0
    rc_ = tp/(tp+fn_) if (tp+fn_)>0 else 0
    f1_ = 2*pr_*rc_/(pr_+rc_) if (pr_+rc_)>0 else 0
    print(f"{thr:<12.2f} {pr_:<12.4f} {rc_:<12.4f} {f1_:<10.4f} {tp}")

print("\nTop-K Lift (vs base fraud rate):")
print(f"{'K%':<8} {'Top-K N':<10} {'Precision@K':<14} {'Base Rate':<12} {'Lift':<10}")
print("-"*60)
for kf in [0.01, 0.02, 0.05, 0.10]:
    k_n, p_at_k, base_rate, lift = topk_lift(all_labels, all_probs_test, k_frac=kf)
    print(f"{int(kf*100):<8} {k_n:<10} {p_at_k:<14.4f} {base_rate:<12.4f} {lift:<10.3f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(rec_v, prec_v, lw=2, label=f'PR-AUC={pr_auc:.3f}')
axes[0].set(title='Precision-Recall Curve', xlabel='Recall', ylabel='Precision')
axes[0].legend()
axes[0].grid(alpha=0.3)

if 'pseudo_confidence_history' in globals() and len(pseudo_confidence_history) > 0:
    conf_df = pd.DataFrame(pseudo_confidence_history)
    axes[1].plot(conf_df['round'], conf_df['mean_confidence'], marker='o', label='Mean confidence')
    axes[1].fill_between(
        conf_df['round'],
        conf_df['p25_confidence'],
        conf_df['p75_confidence'],
        alpha=0.2,
        label='IQR (P25-P75)'
    )
    axes[1].plot(conf_df['round'], conf_df['p50_confidence'], linestyle='--', label='Median confidence')
    axes[1].set(
        title='Stability Over Pseudo-Label Rounds',
        xlabel='Pseudo-label round',
        ylabel='Confidence |2p-1| (0=fuzzy, 1=sharp)',
        ylim=(0, 1)
    )
    axes[1].legend()
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No pseudo-label confidence history\nRun training cell first.', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.savefig('model_evaluation_sage.png', dpi=150, bbox_inches='tight')
print("\n✓ Plots saved to model_evaluation_sage.png")
plt.show()


NameError: name 'data_graph' is not defined

# Leave-One-Feature-Out (LOFO) Analysis

This section evaluates feature usefulness **without retraining**:
- Keep the trained model fixed
- For each customer feature, set it to a constant value (0 in standardized space)
- Recompute test metrics and compare against baseline

Metrics used here:
- **PR-AUC** (better for imbalanced fraud detection)
- **Top-5% Lift** (how concentrated fraud is in the highest-risk segment)

A larger drop in PR-AUC / Top-5% Lift indicates a more important feature.

In [ ]:
from sklearn.metrics import precision_recall_curve, auc, confusion_matrix
from torch_geometric.loader import NeighborLoader
import pandas as pd
import numpy as np
import torch

# Feature groups aligned with the current 16-feature CUSTOMER_FEATURE_COLS
KYC_FEATURES = ['age', 'income', 'tenure', 'sales', 'entity_type']
TXN_FEATURES = [
    'avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'txn_count',
    'cash_rate', 'ecom_rate', 'avg_24h_velocity',
]
AE_FEATURES = ['avg_txn_ae_error', 'neighborhood_risk']
METAPATH_FEATURES = ['metapath_pair_density_mean', 'metapath_pair_density_max']

def topk_lift(y_true, y_prob, k_frac=0.05):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    k = max(1, int(np.ceil(n * k_frac)))
    top_idx = np.argsort(-y_prob)[:k]
    base_rate = y_true.mean() if n > 0 else 0.0
    precision_at_k = y_true[top_idx].mean() if k > 0 else 0.0
    lift = (precision_at_k / base_rate) if base_rate > 0 else 0.0
    return int(k), float(precision_at_k), float(base_rate), float(lift)

def evaluate_on_test(graph_obj, model_obj, device_obj):
    test_loader_local = NeighborLoader(
        graph_obj,
        num_neighbors=NUM_NEIGHBORS,
        batch_size=BATCH_SIZE * 2,
        input_nodes=('customer', graph_obj['customer'].test_mask),
        shuffle=False,
        num_workers=0,
    )

    model_obj.eval()
    probs_all, labels_all = [], []
    with torch.no_grad():
        for b in test_loader_local:
            b = b.to(device_obj)
            edge_weight_dict = {et: b[et].edge_weight for et in b.edge_types if 'edge_weight' in b[et]}
            edge_attr_dict = {et: b[et].edge_attr for et in b.edge_types if 'edge_attr' in b[et]}
            out_t = model_obj(b.x_dict, b.edge_index_dict, edge_weight_dict, edge_attr_dict)
            bs_t = b['customer'].batch_size
            probs_t = torch.sigmoid(out_t.squeeze()[:bs_t]).cpu()
            y_t = b['customer'].y[:bs_t].cpu()
            lm_t = y_t != -1
            if lm_t.sum() > 0:
                probs_all.extend(probs_t[lm_t].tolist())
                labels_all.extend(y_t[lm_t].tolist())

    probs_all = np.array(probs_all, dtype=float)
    labels_all = np.array(labels_all, dtype=float)
    preds_all = (probs_all > 0.5).astype(int)

    prec_v, rec_v, _ = precision_recall_curve(labels_all, probs_all)
    pr_auc_val = auc(rec_v, prec_v)

    k5_n, k5_prec, base_rate, top5_lift_val = topk_lift(labels_all, probs_all, k_frac=0.05)

    cm = confusion_matrix(labels_all, preds_all)
    fraud_recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0.0
    fraud_precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0.0
    fraud_f1 = (
        2 * fraud_recall * fraud_precision / (fraud_recall + fraud_precision)
        if (fraud_recall + fraud_precision) > 0
        else 0.0
    )

    return {
        'pr_auc': float(pr_auc_val),
        'top5_lift': float(top5_lift_val),
        'top5_precision': float(k5_prec),
        'base_rate': float(base_rate),
        'top5_count': int(k5_n),
        'fraud_recall': float(fraud_recall),
        'fraud_precision': float(fraud_precision),
        'fraud_f1': float(fraud_f1),
        'n_eval': int(len(labels_all)),
    }

def evaluate_feature_ablation(feature_name, feature_value=0.0):
    if feature_name not in CUSTOMER_FEATURE_COLS:
        raise ValueError(f"Feature {feature_name} not found in CUSTOMER_FEATURE_COLS")

    idx = CUSTOMER_FEATURE_COLS.index(feature_name)
    graph_ab = data_graph.clone()
    x_mod = graph_ab['customer'].x.clone()
    x_mod[:, idx] = float(feature_value)
    graph_ab['customer'].x = x_mod

    m = evaluate_on_test(graph_ab, model_sage, device)
    m['feature'] = feature_name
    m['set_to'] = float(feature_value)
    return m

print('Running baseline test evaluation for LOFO...')
baseline_metrics = evaluate_on_test(data_graph, model_sage, device)
print(
    f"Baseline -> PR-AUC={baseline_metrics['pr_auc']:.4f} | "
    f"Top-5% Lift={baseline_metrics['top5_lift']:.3f} | "
    f"Fraud F1={baseline_metrics['fraud_f1']:.4f}"
)

rows = []
all_features = list(CUSTOMER_FEATURE_COLS)

for feat in all_features:
    res = evaluate_feature_ablation(feat, feature_value=0.0)
    rows.append(res)
    print(
        f"LOFO {feat:<32} "
        f"PR-AUC={res['pr_auc']:.4f} "
        f"Top-5% Lift={res['top5_lift']:.3f} "
        f"Fraud F1={res['fraud_f1']:.4f}"
    )

loo_results_df = pd.DataFrame(rows)
loo_results_df['delta_pr_auc'] = baseline_metrics['pr_auc'] - loo_results_df['pr_auc']
loo_results_df['delta_top5_lift'] = baseline_metrics['top5_lift'] - loo_results_df['top5_lift']
loo_results_df['delta_fraud_f1'] = baseline_metrics['fraud_f1'] - loo_results_df['fraud_f1']

is_kyc = loo_results_df['feature'].isin(KYC_FEATURES)
is_ae = loo_results_df['feature'].isin(AE_FEATURES)
is_metapath = loo_results_df['feature'].isin(METAPATH_FEATURES)
loo_results_df['feature_group'] = np.select(
    [is_kyc, is_ae, is_metapath],
    ['kyc', 'autoencoder', 'metapath'],
    default='transaction_aggregate'
)

loo_results_df = loo_results_df.sort_values(['delta_pr_auc', 'delta_top5_lift'], ascending=False).reset_index(drop=True)
loo_results_df.to_csv('feature_lofo_results.csv', index=False)

print('\nTop features by PR-AUC drop (bigger means more useful):')
print(loo_results_df[['feature', 'feature_group', 'delta_pr_auc', 'delta_top5_lift', 'delta_fraud_f1']].to_string(index=False))
print("\nSaved LOFO metrics -> feature_lofo_results.csv")

print('\nGroup summary:')
print(
    loo_results_df.groupby('feature_group')[['delta_pr_auc', 'delta_top5_lift', 'delta_fraud_f1']]
    .mean()
    .round(6)
    .to_string()
)


Running baseline test evaluation for LOFO...
Baseline -> PR-AUC=0.6543 | Top-5% Lift=1.606 | Fraud F1=0.7585
LOFO age                          PR-AUC=0.6568 Top-5% Lift=1.606 Fraud F1=0.7585
LOFO income                       PR-AUC=0.6551 Top-5% Lift=1.606 Fraud F1=0.7585
LOFO tenure                       PR-AUC=0.6547 Top-5% Lift=1.606 Fraud F1=0.7585
LOFO sales                        PR-AUC=0.6547 Top-5% Lift=1.606 Fraud F1=0.7585
LOFO entity_type                  PR-AUC=0.6547 Top-5% Lift=1.606 Fraud F1=0.7585
LOFO avg_txn_amount               PR-AUC=0.5504 Top-5% Lift=1.223 Fraud F1=0.7270
LOFO max_txn_amount               PR-AUC=0.8744 Top-5% Lift=1.606 Fraud F1=0.7680
LOFO std_txn_amount               PR-AUC=0.6207 Top-5% Lift=1.606 Fraud F1=0.7330
LOFO txn_count                    PR-AUC=0.8990 Top-5% Lift=1.606 Fraud F1=0.4852
LOFO cash_rate                    PR-AUC=0.6543 Top-5% Lift=1.606 Fraud F1=0.7585
LOFO ecom_rate                    PR-AUC=0.6528 Top-5% Lift=1.606 Fraud